In [ ]:
from cropspectai import config
from cropspectai.data_analysis import preprocessing
from cropspectai.modeling.train import train_model

import pandas as pd

from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor


# --------------------------------------------------
# Load data
# --------------------------------------------------

spectral_length = "as7343" # "full" or "1000" or "as7343"
target = "cab" # "cab" or None
indices = "established" # "all" or "established" or None

if spectral_length == "as7343":
    if target == None:
        train_path = (
            config.PROCESSED_PARQUET_DATA_SPLITS_PATH
            / "all"
            / "train.parquet"
        )
    elif target == "cab":
        if indices == "all": # all engineered features
            train_path = (
                        config.PROCESSED_PARQUET_DATA_SPLITS_PATH
                        / target
                        / "train_all_features.parquet"
                    )
        elif indices == "established": # only established new features
            train_path = (
                        config.PROCESSED_PARQUET_DATA_SPLITS_PATH
                        / target
                        / "train_established_features.parquet"
                    ) 
        elif indices == None: # no engineered features
            train_path = (
                        config.PROCESSED_PARQUET_DATA_SPLITS_PATH
                        / target
                        / "train.parquet"
                    )       
elif spectral_length == "full" or "1000":
    train_path = (
        config.RAW_PARQUET_DATA_SPLITS_PATH
        / "train.parquet"
    )

df = pd.read_parquet(train_path)
print("Shape of imported data: ", df.shape)

In [ ]:
# --------------------------------------------------
# Get feature and target column names
# --------------------------------------------------

X, y = preprocessing.split_dataset_into_X_y(df, target)

targets = y.columns.tolist()

print("Targets:")
print(targets)

print("Features:")
print(X.columns.tolist())

## X (features) as indices only

In [ ]:
AS7343_CHANNELS = list(config.AS7343_CHANNELS.keys())


X = X.drop(
        columns=AS7343_CHANNELS,
        errors="ignore",
    ).copy()

features = X.columns.tolist()

print("Features:")
print(features)

print(X.shape)

df = pd.concat([X, y], axis = 1).copy()
print(df.shape)


## Hyperspectra up to 1000nm only

In [ ]:
if spectral_length == "1000":
    print(X.shape)
    X_1000 = X.loc[:, "400":"1000"].copy()
    print(X_1000.shape)
    print(y.shape)

    df = pd.concat([X_1000, y], axis = 1).copy()
    print(df.shape)

In [ ]:
# --------------------------------------------------
# Store results
# --------------------------------------------------

results = {}


# --------------------------------------------------
# Run XGBoost independently for every feature and every target
# --------------------------------------------------

# for feature in features:
for target in targets:

    print("\n" + "=" * 60)
    print(f"TARGET: {target}")
    print("=" * 60)

    # IMPORTANT:
    # Start from the original dataframe for every target
    df_target = preprocessing.drop_duplicates(
        df,
        target
    ).copy()

    # Split into features and targets AFTER target-specific
    # duplicate handling
    X, y = preprocessing.split_dataset_into_X_y(df_target, target)

    # Train / validation split
    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
    )

    # Create a fresh model for every target
    model_xgb = XGBRegressor(
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=1.0,

        tree_method="hist",

        objective="reg:squarederror",
        eval_metric="rmse",

        n_jobs=-1,
        random_state=42,
    )

    # Train
    pipeline_xgb, metrics = train_model(
        model_xgb,
        X_train,
        y_train,
        X_val,
        y_val,
        scale=False,
    )

    metrics["Samples"] = len(df_target)
    results[target] = metrics

In [ ]:
# --------------------------------------------------
# Create results table
# --------------------------------------------------

results_df = pd.DataFrame(results)
print("\nXGBoost Results for each target")
print(results_df)

In [ ]:
# file_name = f"xgboost_results_{target}_{spectral_length}-{indices}-only.csv"
# output_path = config.REPORTS_DIR / file_name

# results_df.to_csv(
#     output_path,
#     index=True,
# )

- full length hyperspectra gives better R2 score for all targets except CAB
- 1000nm cutoff hyperspectra gives consistently lower R2 score than full except for CAB
- as7343 does poor at all targets and consistently lower than 1000nm 
 - but CAB score remains on par with 1000nm
 - more importantly does better than even full

Conclusion:
- CAB is better predicted with data within 1000nm range
- as7343 is better capable at predicting CAB than all other targets